# 02. RAG: 왜 필요한가 + 동작 원리

01에서 LLM을 부르는 법을 봤다. 이제 그걸로 **사내 도메인 질문**에 답하려고 하면 곧장 막힌다. 이 노트북은 그 막힘의 정체를 보여주고, 다음 노트북부터 손으로 만들 RAG의 큰 그림을 잡는다.

## 학습 목표
- LLM 단독으로 도메인 질문에 답하기 어려운 이유를 설명할 수 있다.
- RAG (Retrieval-Augmented Generation)의 인덱싱·질의 2단계 흐름을 그릴 수 있다.
- 이 강의의 나머지 9개 노트북이 RAG의 어느 부품을 만드는지 짚을 수 있다.
- 폐쇄망에서 RAG가 갖는 의미를 안다.

## 핵심 키워드
`Retrieval-Augmented Generation` · `지식 컷오프` · `Hallucination` · `Citation` · `Embedding` · `Vector Search`

## 1. LLM 단독의 한계

LLM은 학습 데이터에 들어 있는 것만 안다. 폐쇄망 금융 도메인에서는 네 가지 한계가 동시에 작동한다.

| 한계 | 예시 | 결과 |
|---|---|---|
| **지식 컷오프** | 학습 시점 이후 시행된 규정 | 모름 / 추측 |
| **사내·도메인 문서 미보유** | 우리 은행 내부 규정, 폐쇄망 위키 | 일반론만 답함 |
| **환각 (Hallucination)** | "○○법 제15조에 따르면…" 그럴듯하지만 거짓 | 잘못된 인용 |
| **출처 추적 불가** | "어디서 가져왔어?" | 알 수 없음 |

### 직접 확인: 도메인 질문을 LLM에 그냥 던져보면

In [ ]:
import sys
sys.path.insert(0, '..')

from common import get_chat_model, provider_badge

print(provider_badge())
llm = get_chat_model(temperature=0.0)

# 강의 후반에 다룰 전자금융감독규정 안의 구체 조항을 물어본다.
question = "전자금융감독규정에서 정한 IT 부문 인력 비율과 정보보호 인력 비율은 각각 몇 %인가? 근거 조항 번호도 알려줘."

print('=== LLM 단독 답변 (컨텍스트 없음) ===')
print(llm.invoke(question).content)

> 🔎 답변을 보면 두 가지 중 하나다.
> - **모른다고 답함** → 정직하지만 무용함.
> - **그럴듯하게 만들어냄** → 더 위험함. 사용자는 사실로 받아들임.
>
> 사람이라면 "규정집을 펴서 해당 조항을 보고 답하라"가 정답이다. RAG는 이걸 자동화한다.

## 2. 컨텍스트를 함께 주면?

같은 질문에 **관련 규정 본문**을 직접 넣어 본다. 즉, "책의 해당 페이지를 펴서 보여주는" 일을 사람이 손으로 한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 전자금융감독규정에서 직접 발췌한 조항 본문
context = '''
[전자금융감독규정 제8조 (인력)]
① 금융회사 또는 전자금융업자는 정보기술부문 인력을 총 임직원수의 100분의 5 이상,
정보보호인력은 정보기술부문 인력의 100분의 5 이상 확보하여야 한다.
'''

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 금융 규정 Q&A 어시스턴트입니다. '
     '반드시 주어진 컨텍스트에 근거해서만 답하고, 근거 조항을 함께 인용하세요. '
     '컨텍스트에 없는 내용은 "제공된 자료로는 알 수 없습니다"라고 답하세요.'),
    ('human', '컨텍스트:\n{context}\n\n질문: {question}'),
])

chain = rag_prompt | llm | StrOutputParser()

print('=== 컨텍스트 + LLM 답변 ===')
print(chain.invoke({'context': context, 'question': question}))

답이 **정확해지고 + 근거 조항이 함께 나온다**. 이게 RAG의 핵심 아이디어다.

> **RAG (Retrieval-Augmented Generation)**:
> 질문에 답하기 전에 **관련 문서를 검색해서 프롬프트에 주입**하는 패턴.

실무에서는 사람이 컨텍스트를 손으로 못 끼워주므로 **자동화된 검색기**가 그 자리를 맡는다.

## 3. RAG의 동작 흐름

```
┌────────────── INDEXING (사전 1회) ───────────────┐
│                                                  │
│  원본 문서 → 로더 → 청킹 → 임베딩 → 벡터스토어    │
│             (03)   (03)   (04)     (04)          │
│                                                  │
└──────────────────────────────────────────────────┘

┌──────────────── QUERY (매번 실행) ───────────────┐
│                                                  │
│  질문 → 임베딩 → 벡터검색 → 상위 K개 컨텍스트     │
│                  (05·06·07)                      │
│       → 프롬프트 결합 → LLM → 답변 + 인용        │
│              (08)        (01의 LCEL 패턴)        │
│                                                  │
└──────────────────────────────────────────────────┘
```

핵심 비대칭: **인덱싱은 무겁고 한 번만, 질의는 가볍게 매번**. 그래서 청킹과 임베딩 품질이 전체 시스템의 천장을 결정한다.

## 4. 이 강의 로드맵 — 각 노트북이 만들 부품

| # | 노트북 | 만드는 부품 |
|---|---|---|
| 01 | LangChain 기본 | RAG의 **생성기**(prompt · LLM · parser) |
| **02** | **이 노트북** | **개념과 큰 그림** |
| 03 | 문서 로더·청킹 | 인덱싱 ① 원본 → 청크 |
| 04 | 벡터스토어 비교 | 인덱싱 ② 청크 → 벡터 인덱스 |
| 05 | Retriever 패턴 | 질의 ① 검색 모드 (similarity / MMR / filter) |
| 06 | Hybrid · 앙상블 | 질의 ② 검색 고도화 — recall (BM25 + Dense) |
| 07 | Reranking | 질의 ③ 검색 고도화 — precision (Cross-Encoder) |
| 08 | RAG 파이프라인 | **모든 부품을 LCEL로 한 체인으로 묶기** |
| 09 | RAGAS 평가 | 체인 품질 측정 |
| 10 | Agent · LangGraph | RAG를 **도구**로 쓰는 추론 루프 |
| 11 | Streamlit 캡스톤 | 완성형 웹 앱 |

## 5. 폐쇄망 RAG의 의미

| 구성요소 | 외부 API 의존 (☁️) | 폐쇄망 대안 (🔒) |
|---|---|---|
| LLM | OpenAI / Anthropic API | Ollama + Qwen / Llama / Gemma |
| Embedding | OpenAI Embeddings | BGE-M3, multilingual-e5 등 로컬 |
| Vector Store | Pinecone, Weaviate Cloud | Chroma / FAISS / Qdrant 로컬 |
| Reranker | Cohere Rerank | BGE-reranker-v2-m3 로컬 |

→ **모든 데이터·연산이 망 내부에 머문다**는 점이, 금융·국방·의료 같은 분야에서 RAG로 사내 LLM 시스템을 만드는 가장 큰 동기다.

---

다음 노트북(`03_loader_chunking.ipynb`)에서 인덱싱 단계의 첫 부품인 **문서 로더와 청킹**부터 손으로 만든다.